# A covariance square root by matrix Newton iteration
A target covariance is fixed in the laboratory axes. Start with a positive matrix whose axes are tilted, use the full matrix derivative, and compare the result with a spectral reference. NumPy and Matplotlib suffice.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

anisotropy = 9.0              # larger eigenvalue of target, smaller one is 1
initial_angle = 35.0         # tilt of the starting estimate in degrees
assert 1 <= anisotropy <= 100 and 0 <= initial_angle <= 90
A = np.diag([float(anisotropy), 1.0])
a = np.deg2rad(initial_angle)
rotation = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
X0 = rotation @ np.diag([2., 1.]) @ rotation.T
target_root = np.diag([np.sqrt(anisotropy), 1.0])

## Solve the Sylvester equation in the current eigenbasis
The derivative of the square at X maps H to XH+HX. In the eigenbasis of a positive X each entry of that sum is an entry of H times a sum of two positive eigenvalues.

In [ ]:
def correction(X, residual):
    d, Q = np.linalg.eigh(X)
    if np.min(d) <= 0:
        raise ValueError("The chosen positive branch has been left.")
    transformed = Q.T @ residual @ Q
    return Q @ (transformed / (d[:,None] + d[None,:])) @ Q.T

history = [X0.copy()]
for _ in range(30):
    X = history[-1]
    R = A-X@X
    if np.linalg.norm(R) <= 3e-14*np.linalg.norm(A):
        break
    H = correction(X, R)
    history.append(X+H)
    assert np.allclose(H@X + X@H, R, atol=2e-12)
assert np.allclose(history[-1], target_root, atol=2e-10)
wrong = 0.5*np.linalg.solve(X0, A-X0@X0)
print("Initial residual of scalar-style derivative solve:", np.linalg.norm(X0@wrong+wrong@X0-(A-X0@X0)))
print("Positive square root:")
print(history[-1])

## View the iteration as a change of ellipse
A unit circle transformed by the root gives a one-standard-deviation covariance contour (its probability content depends on dimension). The right-hand graph charts the relative residual. Finite precision sets the visible floor.

In [ ]:
fig, (ellipse, graph) = plt.subplots(1, 2, figsize=(10, 4.0), layout="constrained")
theta = np.linspace(0, 2*np.pi, 241)
circle = np.array([np.cos(theta), np.sin(theta)])
for X, name, color, style in [(history[0], "initial", "#8b6816","--"), (history[min(1,len(history)-1)], "first iteration", "#b94626",":"), (target_root, "required", "#145c71","-")]:
    coords = X@circle
    ellipse.plot(coords[0], coords[1], style, color=color, label=name)
ellipse.set(aspect="equal", title="Action on a unit circle", xlabel="First coordinate", ylabel="Second coordinate")
ellipse.legend(fontsize=9)
errors = [np.linalg.norm(A-X@X)/np.linalg.norm(A) for X in history]
graph.semilogy(range(len(errors)), np.maximum(errors, 2e-16), "o-", color="#145c71")
graph.set(title="Residual of the matrix equation", xlabel="Accepted iteration", ylabel="Relative Frobenius residual")
for ax in (ellipse, graph): ax.grid(alpha=.2)
plt.show()

## Verify the derivative by vectorising it
Column stacking converts the Sylvester equation into a four-by-four linear system. Compare the two constructions. In a local OneSimulation environment the explicit system uses its dense linear solver; elsewhere NumPy solves it.

In [ ]:
try:
    from onesim.numerics.linear.matrices.dense_linear_algebra import solve_dense as solve
except ImportError:
    solve = np.linalg.solve
R = A-X0@X0
K = np.kron(np.eye(2), X0) + np.kron(X0.T, np.eye(2))
H = solve(K, R.ravel(order="F")).reshape((2,2), order="F")
assert np.allclose(H, correction(X0, R))
print("Independent Sylvester check:", np.linalg.norm(K@H.ravel(order="F")-R.ravel(order="F")))